# 步骤四：大语言模型（LLM）评估

支持 **Ollama（本地）** 和 **OpenAI 兼容 API（云端智谱AI）** 两种调用方式，对 AI 生成代码进行代码审查，与 lab4 人类 PR 结果对比。

**切换方式：** 修改 `config.py` 中的 `LLM_PROVIDER` 为 `"ollama"` 或 `"openai"`

**两个任务：**
- Task 1: Merge Prediction（是否应该合并）
- Task 2: Review Comment Generation（代码审查评论）

**4 种 Prompt 类型：**
- zero_shot / few_shot / cot / role_based

**4 种上下文类型：**
- diff_only / diff_pr_desc / diff_commit / diff_extra

In [1]:
import sys
sys.path.insert(0, '..')
sys.path.append('../../lab4')

import importlib
import config
importlib.reload(config)

<module 'config' from 'f:\\学习资料\\智能软件工程实践\\lab5\\step4_llm\\..\\config.py'>

## 1. 数据准备

加载 step1 筛选出的 AI PR，提取字段，构建 4 种上下文。

In [2]:
from data_prepare import AIPRDataPreparer

preparer = AIPRDataPreparer()
contexts, summary = preparer.run_all()

print(f"\n数据摘要:")
print(f"  PR 总数: {summary['total_prs']}")
print(f"  Merged: {summary['merged']}")
print(f"  Not Merged: {summary['not_merged']}")
print(f"  平均 diff 长度: {summary['avg_diff_length']:.0f} 字符")

步骤四数据准备: 加载 AI PR → 提取字段 → 构建上下文

[1/4] 加载 AI PR 数据...
  加载 kubernetes_kubernetes: 5 个 AI PR
  加载 pytorch_pytorch: 84 个 AI PR
  加载 tensorflow_tensorflow: 3 个 AI PR
  加载 microsoft_vscode: 183 个 AI PR
  加载 langchain-ai_langchain: 69 个 AI PR
  共加载 344 个 AI PR

[2/4] 随机采样 100 条 PR...
  采样完成，共 100 条

[3/4] 提取字段...
  提取完成，共 100 条

[4/4] 构建 4 种上下文...
  已处理 50/100 条...
  已处理 100/100 条...
  上下文构建完成，共 100 条

摘要: 100 个 PR, merged=41, not_merged=59

数据摘要:
  PR 总数: 100
  Merged: 41
  Not Merged: 59
  平均 diff 长度: 166220 字符


## 2. LLM 推理 + 评估

调用 LLM 对每条 AI PR 进行推理，然后评估指标。

**Ollama 模式：** 需要先启动 Ollama 服务 `ollama serve`

**OpenAI 模式：** 需要在 `.env` 文件中配置 `ZHIPUAI_API_KEY=你的密钥`

In [3]:
from evaluate import LLMEvaluator

evaluator = LLMEvaluator()
results = evaluator.run_all()

evaluator.print_summary()

步骤四: LLM 代码审查测试
  Provider: openai
加载上下文缓存: f:\学习资料\智能软件工程实践\lab5\data\step4_contexts.json
  已加载 100 条上下文
已加载已有结果: 2830 条记录

组 1/8: merge_prediction + zero_shot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\merge_prediction_zero_shot.json


[merge_prediction][zero_shot]: 100%|███████████| 400/400 [00:00<00:00, 770.56req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=346, No=52, Unknown=2
  Accuracy : 0.5000
  Precision: 0.4480
  Recall   : 0.9509
  F1-score : 0.6090
  ROC-AUC  : 0.5691

组 2/8: merge_prediction + few_shot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\merge_prediction_few_shot.json


[merge_prediction][few_shot]: 100%|████████████| 400/400 [00:00<00:00, 819.82req/s, new=0, skip=400]


  跳过 400 条已有记录
  加载已有 400 条，新增 0 条
  本组完成: 400 条记录

  预测分布: Yes=349, No=51, Unknown=0
  Accuracy : 0.4975
  Precision: 0.4470
  Recall   : 0.9512
  F1-score : 0.6082
  ROC-AUC  : 0.5667

组 3/8: merge_prediction + cot
  文件: f:\学习资料\智能软件工程实践\lab5\results\step4\merge_prediction_cot.json


...

  [OpenAI 调用失败] 429 Too Many Requests (Retry-After: N/A)，1s 后重试 (1/3)
  [OpenAI 调用失败] 429 Too Many Requests (Retry-After: N/A)，2s 后重试 (2/3)


[merge_prediction][role_based]:  18%|▏| 72/400 [25:28<3:08:20, 34.45s/req, new=42, lat=43.4s, skip=3

  [OpenAI 调用失败] 429 Too Many Requests (Retry-After: N/A)，1s 后重试 (1/3)


[merge_prediction][role_based]:  20%|▏| 78/400 [30:37<2:06:26, 23.56s/req, new=48, lat=36.1s, skip=3


KeyboardInterrupt: 

## 3. 断点续传

中断后重新运行 `evaluator.run_all()` 即可自动跳过已完成记录，从断点继续。

结果按 `(task, prompt_type)` 保存到独立文件：
- `lab5/results/step4/merge_prediction_zero_shot.json`
- `lab5/results/step4/merge_prediction_few_shot.json`
- ... 共 8 个文件